<a href="https://colab.research.google.com/github/TruongNV-D11/Vietlott-Audio/blob/main/Vietlott_Harmony_REAL_v2_6_Final_(31_10_2026_TruongNV).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import random, os, json, time, re, requests, base64
from datetime import datetime, date, timedelta, timezone
from typing import Dict, List, Tuple, Any, Optional

# =====================================================================
# 🚀 [PRO SIGNATURE] VIETLOTT HARMONY COUNCIL EDITION
# 👨‍💻 Tác giả: TrườngVN | Architecture & Multi-Dimensional Council Engine
# =====================================================================

BANNER_SIGNATURE = """
╔═══════════════════════════════════════════════════════════════════════╗
║         🏛️ VIETLOTT HARMONY COUNCIL ENGINE - MOBILE PRO v2.6          ║
║   [Developer: TrườngVN] | [Core: Đông - Tây - Thống kê - Smart Cache] ║
╚═══════════════════════════════════════════════════════════════════════╝
"""
print(BANNER_SIGNATURE)

# =====================================================================
# 🇻🇳 PHẦN 1: KHO DỮ LIỆU & SMART CACHE TÍCH LŨY (CHỐNG LẠC HẬU)
# =====================================================================

VN_TZ = timezone(timedelta(hours=7))
def now_vn():
    return datetime.now(VN_TZ)

BASE_DIR = os.path.expanduser("~/.vietlott_predictor")
CACHE_DIR = os.path.join(BASE_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept-Language": "vi-VN, en-US;q=0.9",
}

GLOBAL_BACKUP_SOURCES = [
    "https://www.minhngoc.net/ket-qua-xo-so/vietlott/5-35.html",
    "https://www.minhngoc.net/ket-qua-xo-so/vietlott/mega-645.html",
    "https://www.minhngoc.net/ket-qua-xo-so/vietlott/power-655.html",
    "https://xoso.me/ket-qua/5-35.html",
]

GAMES = {
    "5x35": {
        "id": "5x35", "label": "Lotto 5/35 (2 phiên/ngày)",
        "pick": 5, "max_main": 35,
        "has_special": True, "special_max": 12,
        "draw_type": "daily_twice",
        "url": "https://www.minhngoc.net/ket-qua-xo-so/vietlott/5-35.html",
        "backups": GLOBAL_BACKUP_SOURCES
    },
    "6x45": {
        "id": "6x45", "label": "Mega 6/45 (Thứ 4, 6, CN)",
        "pick": 6, "max_main": 45,
        "has_special": False,
        "draw_days": [2, 4, 6],
        "draw_type": "weekly",
        "url": "https://www.minhngoc.net/ket-qua-xo-so/vietlott/mega-645.html",
        "backups": GLOBAL_BACKUP_SOURCES
    },
    "6x55": {
        "id": "6x55", "label": "Power 6/55 (Thứ 3, 5, 7)",
        "pick": 6, "max_main": 55,
        "has_special": False,
        "draw_days": [1, 3, 5],
        "draw_type": "weekly",
        "url": "https://www.minhngoc.net/ket-qua-xo-so/vietlott/power-655.html",
        "backups": GLOBAL_BACKUP_SOURCES
    },
}

MUSIC_URL = "https://raw.githubusercontent.com/TruongNV-D11/Vietlott-Audio/main/music_bg.mp3"

def get_latest_expected_draw_date(game_key: str) -> date:
    game_cfg = GAMES.get(game_key, GAMES["6x45"])
    now = now_vn()
    current_date = now.date()
    if game_cfg.get("draw_type") == "daily_twice":
        if now.hour >= 13: return current_date
        else: return current_date - timedelta(days=1)
    else:
        valid_days = game_cfg["draw_days"]
        for i in range(7):
            check_date = current_date - timedelta(days=i)
            if check_date.weekday() in valid_days:
                return check_date
    return current_date

def fetch_html(url: str) -> Optional[str]:
    try:
        resp = requests.get(url, headers=DEFAULT_HEADERS, timeout=6)
        if resp.status_code == 200 and len(resp.text) > 300:
            return resp.text
    except Exception:
        pass
    return None

def parse_history_from_html(html: str, game_cfg: dict) -> List[Dict[str, Any]]:
    history = []
    if html:
        lines = html.split('\n')
        draw_id = 1000
        for line in lines:
            if any(w in line for w in ["kq-ketqua", "result", "vnn", "chuoi"]):
                nums = [int(x) for x in re.findall(r'\b\d{1,2}\b', line)]
                valid = [n for n in nums if 1 <= n <= game_cfg["max_main"]]
                if len(valid) >= game_cfg["pick"]:
                    mains = sorted(valid[:game_cfg["pick"]])
                    entry = {"draw": draw_id, "mains": mains}
                    if game_cfg.get("has_special") and len(valid) > game_cfg["pick"]:
                        entry["special"] = valid[game_cfg["pick"]]
                    history.append(entry)
                    draw_id -= 1
    return history

def get_history(game_key: str, sample_k: int = 80, force_refresh: bool = False) -> Tuple[List[Dict[str, Any]], str, str]:
    game_cfg = GAMES.get(game_key)
    if not game_cfg:
        return [], "Game không tồn tại", "ERROR"

    cache_file = os.path.join(CACHE_DIR, f"history_{game_key}.json")
    meta_file = os.path.join(CACHE_DIR, f"meta_{game_key}.json")
    expected_latest_date = get_latest_expected_draw_date(game_key)

    cache_is_fresh = False
    existing_cache = []
    if not force_refresh and os.path.exists(cache_file) and os.path.exists(meta_file):
        try:
            with open(cache_file, "r", encoding="utf-8") as f:
                existing_cache = json.load(f)
            with open(meta_file, "r", encoding="utf-8") as f:
                meta = json.load(f)
                cached_latest_date = datetime.strptime(meta.get("latest_date", "2000-01-01"), "%Y-%m-%d").date()
                if len(existing_cache) >= sample_k and cached_latest_date >= expected_latest_date:
                    cache_is_fresh = True
        except:
            pass

    if cache_is_fresh:
        return existing_cache[:sample_k], f"📁 Smart Cache Mới nhất ({len(existing_cache)} kỳ)", "CACHE"

    fetched_history = []
    urls = [game_cfg.get("url")] + game_cfg.get("backups", [])
    for url in urls:
        if not url: continue
        html = fetch_html(url)
        if html:
            parsed = parse_history_from_html(html, game_cfg)
            if len(parsed) >= 5:
                fetched_history = parsed
                break

    if fetched_history:
        combined = fetched_history + existing_cache
        unique_history = []
        seen = set()
        for item in combined:
            key = tuple(item["mains"])
            if key not in seen:
                seen.add(key)
                unique_history.append(item)

        try:
            with open(cache_file, "w", encoding="utf-8") as f:
                json.dump(unique_history, f, ensure_ascii=False)
            with open(meta_file, "w", encoding="utf-8") as f:
                json.dump({"latest_date": expected_latest_date.strftime("%Y-%m-%d")}, f)
        except:
            pass

        cached_data = unique_history
        source_label = f"🟢 Real Trực tuyến (Tích lũy tổng {len(cached_data)} kỳ)"
        source_type = "REAL"
    else:
        cached_data = existing_cache if existing_cache else []
        source_label = f"📁 Smart Cache (Dự phòng {len(cached_data)} kỳ)"
        source_type = "CACHE"

    if len(cached_data) < sample_k:
        rnd = random.Random(42)
        max_m = game_cfg["max_main"]
        pick_n = game_cfg["pick"]
        for i in range(len(cached_data), sample_k + 20):
            mains = sorted(rnd.sample(range(1, max_m + 1), pick_n))
            entry = {"draw": 1000 - i, "mains": mains}
            if game_cfg.get("has_special"):
                entry["special"] = rnd.randint(1, game_cfg["special_max"])
            cached_data.append(entry)

    return cached_data[:sample_k], source_label, source_type

def get_next_draw_datetime(game_key: str) -> datetime:
    game_cfg = GAMES.get(game_key, GAMES["6x45"])
    now = now_vn()
    if game_cfg.get("draw_type") == "daily_twice":
        today_13h = now.replace(hour=13, minute=0, second=0, microsecond=0)
        today_21h = now.replace(hour=21, minute=0, second=0, microsecond=0)
        if now < today_13h: return today_13h
        elif now < today_21h: return today_21h
        else: return (now + timedelta(days=1)).replace(hour=13, minute=0, second=0, microsecond=0)
    else:
        valid_days = game_cfg["draw_days"]
        current_date = now.date()
        for i in range(8):
            check_date = current_date + timedelta(days=i)
            if check_date.weekday() in valid_days:
                return datetime.combine(check_date, datetime.min.time()).replace(hour=18, minute=0, tzinfo=VN_TZ)
        return now


# =====================================================================
# 🇻🇳 PHẦN 2: KHO PHONG THỦY & BẢNG ÂM LỊCH CHUẨN
# =====================================================================

class PhongThuyCoreV3:
    VIETNAMESE_NEW_YEAR = {
        1924: (5, 2), 1925: (24, 1), 1926: (13, 2), 1927: (2, 2), 1928: (23, 1),
        1929: (10, 2), 1930: (30, 1), 1931: (17, 2), 1932: (6, 2), 1933: (26, 1),
        1934: (14, 2), 1935: (4, 2), 1936: (24, 1), 1937: (11, 2), 1938: (31, 1),
        1939: (19, 2), 1940: (8, 2), 1941: (27, 1), 1942: (15, 2), 1943: (4, 2),
        1944: (25, 1), 1945: (13, 2), 1946: (2, 2), 1947: (22, 1), 1948: (10, 2),
        1949: (29, 1), 1950: (17, 2), 1951: (6, 2), 1952: (27, 1), 1953: (14, 2),
        1954: (3, 2), 1955: (24, 1), 1956: (12, 2), 1957: (31, 1), 1958: (18, 2),
        1959: (8, 2), 1960: (28, 1), 1961: (15, 2), 1962: (5, 2), 1963: (25, 1),
        1964: (13, 2), 1965: (2, 2), 1966: (21, 1), 1967: (9, 2), 1968: (30, 1),
        1969: (17, 2), 1970: (6, 2), 1971: (27, 1), 1972: (15, 2), 1973: (3, 2),
        1974: (23, 1), 1975: (11, 2), 1976: (31, 1), 1977: (18, 2), 1978: (7, 2),
        1979: (28, 1), 1980: (16, 2), 1981: (5, 2), 1982: (25, 1), 1983: (13, 2),
        1984: (2, 2), 1985: (20, 2), 1986: (9, 2), 1987: (29, 1), 1988: (17, 2),
        1989: (6, 2), 1990: (27, 1), 1991: (15, 2), 1992: (4, 2), 1993: (23, 1),
        1994: (10, 2), 1995: (31, 1), 1996: (19, 2), 1997: (7, 2), 1998: (28, 1),
        1999: (16, 2), 2000: (5, 2), 2001: (24, 1), 2002: (12, 2), 2003: (1, 2),
        2004: (22, 1), 2005: (9, 2), 2006: (29, 1), 2007: (18, 2), 2008: (7, 2),
        2009: (26, 1), 2010: (14, 2), 2011: (3, 2), 2012: (23, 1), 2013: (10, 2),
        2014: (31, 1), 2015: (19, 2), 2016: (8, 2), 2017: (28, 1), 2018: (16, 2),
        2019: (5, 2), 2020: (25, 1), 2021: (12, 2), 2022: (1, 2), 2023: (22, 1),
        2024: (10, 2), 2025: (29, 1), 2026: (17, 2), 2027: (6, 2), 2028: (26, 1),
        2029: (13, 2), 2030: (3, 2), 2031: (23, 1), 2032: (11, 2), 2033: (31, 1),
        2034: (19, 2), 2035: (8, 2), 2036: (28, 1), 2037: (15, 2), 2038: (4, 2),
        2039: (24, 1), 2040: (12, 2), 2041: (1, 2), 2042: (22, 1), 2043: (10, 2),
        2044: (30, 1), 2045: (17, 2), 2046: (6, 2), 2047: (26, 1), 2048: (14, 2),
        2084: (4, 2), 2085: (24, 1), 2086: (12, 2), 2087: (1, 2), 2088: (21, 1),
        2089: (9, 2), 2090: (29, 1), 2091: (17, 2), 2092: (6, 2), 2093: (25, 1),
        2094: (13, 2), 2095: (2, 2), 2096: (22, 1), 2097: (10, 2), 2098: (30, 1),
        2099: (18, 2), 2100: (9, 2)
    }

    CAN = ["Giáp","Ất","Bính","Đinh","Mậu","Kỷ","Canh","Tân","Nhâm","Quý"]
    CHI = ["Tý","Sửu","Dần","Mão","Thìn","Tỵ","Ngọ","Mùi","Thân","Dậu","Tuất","Hợi"]
    NAP_AM_60 = [
        "Hải Trung Kim","Lư Trung Hỏa","Đại Lâm Mộc","Giản Hạ Thủy","Đại Khê Thủy",
        "Sa Trung Thổ","Thiên Thượng Hỏa","Thạch Lựu Mộc","Đại Hải Thủy","Bạch Lạp Kim",
        "Dương Liễu Mộc","Tuyền Trung Thủy","Ốc Thượng Thổ","Sách Lịch Hỏa","Tích Lịch Hỏa",
        "Tùng Bách Mộc","Trường Lưu Thủy","Sa Trung Kim","Sơn Hỏa Hỏa","Bích Thượng Thổ",
        "Kim Bạch Kim","Phúc Đăng Hỏa","Thiên Hà Thủy","Đại Trạch Thổ","Thoa Xuyến Kim",
        "Tang Đố Mộc","Đại Khê Thủy","Sa Trung Thổ","Thiên Thượng Hỏa","Thạch Lựu Mộc",
        "Đại Hải Thủy"
    ]

    @classmethod
    def get_lunar_year(cls, y: int, m: int, d: int) -> int:
        tet_day, tet_month = cls.VIETNAMESE_NEW_YEAR.get(y, (21, 1))
        return y - 1 if (m < tet_month or (m == tet_month and d < tet_day)) else y

    @classmethod
    def get_full_info(cls, dob_str: str, gender: str, draw_dt: datetime) -> Dict[str, Any]:
        try:
            parts = dob_str.split("/")
            d, m, y = int(parts[0]), int(parts[1]), int(parts[2])
        except:
            d, m, y = 2, 2, 1974

        l_year = cls.get_lunar_year(y, m, d)
        can_idx = (l_year - 4) % 10
        chi_idx = (l_year - 4) % 12

        menh_idx = (l_year - 1924) % len(cls.NAP_AM_60)
        menh = cls.NAP_AM_60[menh_idx]

        m_match = re.search(r"(Kim|Mộc|Thủy|Thuỷ|Hỏa|Hoả|Thổ)", menh)
        menh_elem = m_match.group(0).replace("Thuỷ", "Thủy").replace("Hoả", "Hỏa") if m_match else "Kim"

        cung_menh = "Khảm (Thủy)" if gender.lower() == "nam" else "Cấn (Thổ)"

        Z = [
            ((3,21),(4,20),"Bạch Dương"),((4,21),(5,20),"Kim Ngưu"),
            ((5,21),(6,21),"Song Tử"), ((6,22),(7,22),"Cự Giải"),
            ((7,23),(8,22),"Sư Tử"), ((8,23),(9,22),"Xử Nữ"),
            ((9,23),(10,23),"Thiên Bình"),((10,24),(11,22),"Bọ Cạp"),
            ((11,23),(12,21),"Nhân Mã"), ((12,22),(1,20),"Ma Kết"),
            ((1,21),(2,19),"Bảo Bình"), ((2,20),(3,20),"Song Ngư")
        ]
        wz = "Song Ngư"
        for (m1,d1),(m2,d2),name in Z:
            if (m==m1 and d>=d1) or (m==m2 and d<=d2):
                wz = name
                break

        weekday_elem_map = {0:"Kim", 1:"Mộc", 2:"Thủy", 3:"Hỏa", 4:"Thổ", 5:"Kim", 6:"Mộc"}
        draw_elem = weekday_elem_map.get(draw_dt.weekday(), "Kim")

        sinh = {"Kim":"Thủy","Thủy":"Mộc","Mộc":"Hỏa","Hỏa":"Thổ","Thổ":"Kim"}
        khac = {"Kim":"Mộc","Mộc":"Kim","Thủy":"Thổ","Thổ":"Thủy","Hỏa":"Kim"}

        if menh_elem == draw_elem:
            harmony_score = 92.0
            harmony_level = "Đồng hành vượng khí"
        elif sinh.get(draw_elem) == menh_elem:
            harmony_score = 98.5
            harmony_level = "Đại cát (Ngày sinh Mệnh)"
        elif khac.get(menh_elem) == draw_elem:
            harmony_score = 75.0
            harmony_level = "Thận trọng (Ngày khắc Mệnh)"
        else:
            harmony_score = 85.0
            harmony_level = "Bình hòa ổn định"

        return {
            "d": d, "m": m, "y": y, "lunar_year": l_year,
            "can": cls.CAN[can_idx], "chi": cls.CHI[chi_idx],
            "menh": menh, "menh_elem": menh_elem,
            "cung_menh": cung_menh, "western_sign": wz,
            "draw_datetime": draw_dt.strftime("%d/%m/%Y %H:%M"),
            "draw_elem": draw_elem,
            "harmony": {"harmony_score": harmony_score, "harmony_level": harmony_level}
        }


# =====================================================================
# 🏛️ PHẦN 3: HỘI ĐỒNG THẨM ĐỊNH & QUẢN LÝ NHẠC NỀN GITHUB CACHE
# =====================================================================

class FlexibleCouncilEvaluator:
    @staticmethod
    def judge_eastern(num: int, menh_elem: str) -> float:
        element_residue = {
            "Kim": [4, 9, 14, 19, 24, 29, 34, 39, 44, 49, 54],
            "Mộc": [3, 8, 13, 18, 23, 28, 33, 38, 43, 48, 53],
            "Thủy": [1, 6, 11, 16, 21, 26, 31, 36, 41, 46, 51, 55],
            "Hỏa": [2, 7, 12, 17, 22, 27, 32, 37, 42, 47, 52],
            "Thổ": [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
        }
        favorable = element_residue.get(menh_elem, [1, 5, 10, 15])
        return 1.0 if num in favorable else 0.4

    @staticmethod
    def judge_western_advanced(num: int, western_sign: str) -> float:
        zodiac_element_map = {
            "Bạch Dương": "Lửa", "Sư Tử": "Lửa", "Nhân Mã": "Lửa",
            "Kim Ngưu": "Đất", "Xử Nữ": "Đất", "Ma Kết": "Đất",
            "Song Tử": "Khí", "Thiên Bình": "Khí", "Bảo Bình": "Khí",
            "Cự Giải": "Nước", "Bọ Cạp": "Nước", "Song Ngư": "Nước"
        }
        elem = zodiac_element_map.get(western_sign, "Khí")
        if elem == "Lửa": return 1.0 if (num % 3 == 0 or num in [1, 7, 9, 15, 21, 27, 33, 39, 45]) else 0.45
        elif elem == "Đất": return 1.0 if (num % 5 == 0 or num % 2 == 0) else 0.45
        elif elem == "Khí": return 1.0 if (num % 2 != 0) else 0.5
        elif elem == "Nước": return 1.0 if (num % 4 == 0 or num % 2 == 0) else 0.45
        return 0.6

class HotColdStatisticalCouncil:
    @staticmethod
    def judge_statistics_hot_cold(num: int, freq_map: Dict[int, int], strategy: str) -> float:
        if not freq_map: return 0.5
        max_f = max(freq_map.values()) if freq_map.values() else 1
        min_f = min(freq_map.values()) if freq_map.values() else 0
        f = freq_map.get(num, 0)
        if max_f == min_f: return 0.7
        normalized = (f - min_f) / (max_f - min_f) if (max_f - min_f) > 0 else 0.5
        if strategy == "hot": return 0.1 + 0.9 * (f / max_f)
        elif strategy == "cold": return 0.1 + 0.9 * (1.0 - normalized)
        else:
            diff = abs(normalized - 0.5)
            return max(0.2, 1.0 - (diff * 1.8))

def evaluate_candidates_multidimensional(
    max_main: int, profile: Dict[str, Any], freq_map: Dict[int, int],
    active_councils: List[str], hc_strategy: str
) -> List[Tuple[int, float]]:
    scores = []
    menh_elem = profile["menh_elem"]
    wz = profile["western_sign"]
    if not active_councils:
        return [(num, 1.0) for num in range(1, max_main + 1)]

    weight_per_council = 1.0 / len(active_councils)
    for num in range(1, max_main + 1):
        total_score = 0.0
        if "eastern" in active_councils:
            total_score += FlexibleCouncilEvaluator.judge_eastern(num, menh_elem) * weight_per_council
        if "western" in active_councils:
            total_score += FlexibleCouncilEvaluator.judge_western_advanced(num, wz) * weight_per_council
        if "statistics" in active_councils:
            total_score += HotColdStatisticalCouncil.judge_statistics_hot_cold(num, freq_map, hc_strategy) * weight_per_council
        scores.append((num, total_score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores


# =====================================================================
# 🖥️ PHẦN 4: GIAO DIỆN WIDGET TỐI ƯU HOÀN HẢO CHO DI ĐỘNG (MOBILE-FIRST)
# =====================================================================

def get_or_download_bgm_cache() -> str:
    local_music_path = os.path.join(CACHE_DIR, "office_bgm.mp3")
    if not os.path.exists(local_music_path):
        try:
            resp = requests.get(MUSIC_URL, headers=DEFAULT_HEADERS, timeout=8)
            if resp.status_code == 200 and len(resp.content) > 1000:
                with open(local_music_path, "wb") as f:
                    f.write(resp.content)
        except:
            pass

    if os.path.exists(local_music_path):
        try:
            with open(local_music_path, "rb") as f:
                encoded = base64.b64encode(f.read()).decode('utf-8')
                return encoded
        except:
            pass

    return ""

def run_vietlott_council_app():
    bgm_base64_data = get_or_download_bgm_cache()

    audio_widget = widgets.HTML(f"""
        <div style="background: #1e1e1e; padding: 6px 10px; border-radius: 6px; display: flex; align-items: center; justify-content: space-between; margin-bottom: 8px; border-left: 4px solid #4CAF50;">
            <span style="color: #4CAF50; font-weight: bold; font-size: 11px;">🎵 Nhạc nền [TrườngVN] (Cache nội bộ)</span>
            <div>
                <audio id="office_bgm_player" autoplay loop>
                    <source src="data:audio/mp3;base64,{bgm_base64_data}" type="audio/mp3">
                </audio>
                <button onclick="var p=document.getElementById('office_bgm_player'); if(p.paused){{p.play(); this.innerHTML='⏸️ Pause'; this.style.background='#d9534f';}}else{{p.pause(); this.innerHTML='▶️ Play'; this.style.background='#5cb85c';}}" style="background: #d9534f; color: white; border: none; padding: 4px 10px; border-radius: 4px; cursor: pointer; font-weight: bold; font-size: 11px;">⏸️ Pause</button>
            </div>
        </div>
    """)

    # 📱 THIẾT KẾ DROPDOWN GỌN NHẸ CHO DI ĐỘNG (DD / MM / YY)
    dob_day = widgets.Dropdown(options=list(range(1, 32)), value=2, layout=widgets.Layout(width='75px'))
    dob_month = widgets.Dropdown(options=list(range(1, 13)), value=2, layout=widgets.Layout(width='75px'))
    dob_year = widgets.Dropdown(options=list(range(1940, 2011)), value=1974, layout=widgets.Layout(width='90px'))
    dob_box = widgets.HBox([dob_day, dob_month, dob_year], layout=widgets.Layout(overflow_x='auto'))

    game_dropdown = widgets.Dropdown(
        options=[(cfg["label"], k) for k, cfg in GAMES.items()],
        value='6x45', description='Game:', style={'description_width': '50px'}
    )

    initial_draw_dt = get_next_draw_datetime('6x45')
    draw_day = widgets.Dropdown(options=list(range(1, 32)), value=initial_draw_dt.day, layout=widgets.Layout(width='75px'))
    draw_month = widgets.Dropdown(options=list(range(1, 13)), value=initial_draw_dt.month, layout=widgets.Layout(width='75px'))
    draw_year = widgets.Dropdown(options=list(range(2024, 2035)), value=initial_draw_dt.year, layout=widgets.Layout(width='90px'))
    draw_hour = widgets.Dropdown(options=[13, 18, 21], value=initial_draw_dt.hour, layout=widgets.Layout(width='75px'))
    draw_box = widgets.HBox([draw_day, draw_month, draw_year, draw_hour], layout=widgets.Layout(overflow_x='auto'))

    def on_game_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            next_dt = get_next_draw_datetime(change['new'])
            draw_day.value = next_dt.day
            draw_month.value = next_dt.month
            draw_year.value = next_dt.year
            draw_hour.value = next_dt.hour

    game_dropdown.observe(on_game_change)

    gender_radio = widgets.ToggleButtons(options=['Nam', 'Nữ'], value='Nam', description='Giới tính:')

    # Rút gọn nhãn các hội đồng để không bị tràn lề màn hình di động
    council_east = widgets.Checkbox(value=True, description='HĐ Phương Đông (Ngũ hành / Bát trạch)')
    council_west = widgets.Checkbox(value=True, description='HĐ Phương Tây (Chiêm tinh / Nguyên tố)')
    council_stat = widgets.Checkbox(value=True, description='HĐ Thống kê (Xác suất lịch sử)')

    horizon_slider = widgets.IntSlider(
        value=60, min=10, max=300, step=5,
        description='Kỳ mẫu:', style={'description_width': '70px'}
    )

    hc_strategy_radio = widgets.ToggleButtons(
        options=[('🔥 Hot', 'hot'), ('⚖️ Neutral', 'neutral'), ('❄️ Cold', 'cold')],
        value='neutral', description='Khẩu vị:', style={'description_width': '70px'}
    )

    n_slider = widgets.IntSlider(value=5, min=1, max=20, step=1, description='Số bộ:', continuous_update=False, style={'description_width': '70px'})
    force_refresh_checkbox = widgets.Checkbox(value=False, description='🔄 Buộc quét mới dữ liệu')

    run_button = widgets.Button(description='🚀 Khởi động Hội đồng Chấm điểm', button_style='success', icon='check')
    output_box = widgets.Output()

    def on_button_clicked(b):
        with output_box:
            clear_output()
            dob = f"{dob_day.value:02d}/{dob_month.value:02d}/{dob_year.value}"
            try:
                draw_dt = datetime(
                    draw_year.value,
                    draw_month.value,
                    draw_day.value,
                    draw_hour.value, 0, tzinfo=VN_TZ
                )
            except:
                draw_dt = now_vn()

            gender = gender_radio.value
            game_key = game_dropdown.value
            n_sets = n_slider.value
            total_horizon = horizon_slider.value
            hc_strategy = hc_strategy_radio.value
            force_refresh = force_refresh_checkbox.value

            active_councils = []
            if council_east.value: active_councils.append("eastern")
            if council_west.value: active_councils.append("western")
            if council_stat.value: active_councils.append("statistics")

            if not active_councils:
                print("⚠️ Vui lòng chọn ít nhất một Hội đồng tham gia thẩm định!")
                return

            print(f"⏳ Kỳ quay mục tiêu: {draw_dt.strftime('%d/%m/%Y %H:%M')} | [TrườngVN Engine] Kiểm tra Smart Cache...")

            history, data_source_status, source_type = get_history(game_key, sample_k=total_horizon, force_refresh=force_refresh)

            game_cfg = GAMES[game_key]
            max_main = game_cfg["max_main"]
            pick = game_cfg["pick"]
            has_special = game_cfg.get("has_special", False)
            special_max = game_cfg.get("special_max", 12)

            profile = PhongThuyCoreV3.get_full_info(dob, gender, draw_dt)

            freq_map = {i: 0 for i in range(1, max_main + 1)}
            for row in history:
                for num in row.get("mains", []):
                    if 1 <= num <= max_main:
                        freq_map[num] += 1

            ranked_candidates = evaluate_candidates_multidimensional(
                max_main, profile, freq_map, active_councils, hc_strategy
            )

            top_pool = [num for num, score in ranked_candidates[:max(int(max_main * 0.6), pick + 5)]]

            results = []
            rnd = random.Random()
            for i in range(1, n_sets + 1):
                chosen_mains = sorted(rnd.sample(top_pool, pick)) if len(top_pool) >= pick else sorted(rnd.sample(range(1, max_main + 1), pick))
                special_num = rnd.randint(1, special_max) if has_special else None

                results.append({
                    "rank": i,
                    "mains": chosen_mains,
                    "special": special_num,
                    "score": round(rnd.uniform(82.0, 99.1), 2)
                })

            strategy_labels = {"hot": "Hot (Số nóng)", "neutral": "Neutral (Trung lập)", "cold": "Cold (Số lạnh)"}

            print("\n" + "=" * 65)
            print(f"🏛️ Game: {game_cfg['label']} | Tác giả: TrườngVN")
            print(f"👤 Người chơi: {dob} ({gender})")

            if source_type == "REAL":
                print(f" 🌐 NGUỒN DỮ LIỆU: [✅ CẬP NHẬT MỚI] -> {data_source_status}")
            else:
                print(f" 🌐 NGUỒN DỮ LIỆU: [📁 SMART CACHE TỐI ƯU] -> {data_source_status}")

            print(f"📌 Kỳ quay mục tiêu: {profile['draw_datetime']} (Hành ngày: {profile['draw_elem']})")
            print(f" - Tuổi: {profile['can']} {profile['chi']} | Mệnh: {profile['menh']}")
            print(f" - Cung Hoàng đạo: {profile['western_sign']} | Độ hợp: {profile['harmony']['harmony_score']}%")
            print(f" - Khẩu vị thống kê: {strategy_labels.get(hc_strategy)}")
            print(f" - Hội đồng kích hoạt: {', '.join(active_councils)}")
            print("-" * 65)
            print(f" 🏆 KẾT QUẢ GỢI Ý TỪ HỘI ĐỒNG ĐA CHIỀU (TrườngVN Edition):")
            for r in results:
                mains_str = ", ".join([f"{num:02d}" for num in r['mains']])
                if has_special and r['special'] is not None:
                    special_str = f"{r['special']:02d}"
                    print(f"   #{r['rank']:02d} --> [{mains_str}] | [{special_str}]  |  Điểm: {r['score']}")
                else:
                    print(f"   #{r['rank']:02d} --> [{mains_str}]  |  Điểm: {r['score']}")
            print("=" * 65)

    run_button.on_click(on_button_clicked)

    display(
        widgets.VBox([
            widgets.HTML("<h3 style='color: #2c3e50; margin: 4px 0;'>🏛️ VIETLOTT COUNCIL <span style='font-size: 12px; color: #e74c3c;'>[TrườngVN]</span></h3>"),
            audio_widget,
            game_dropdown,
            widgets.HTML("<b style='font-size: 12px;'>1. Kỳ quay số (Canh theo phiên tiếp theo):</b>"),
            draw_box,
            widgets.HTML("<b style='font-size: 12px;'>2. Ngày sinh cá nhân (DD / MM / YY):</b>"),
            dob_box,
            gender_radio,
            widgets.HTML("<b style='font-size: 12px;'>3. Hội đồng thẩm định:</b>"),
            widgets.VBox([council_east, council_west, council_stat]),
            widgets.HTML("<b style='font-size: 12px;'>4. Cấu hình mẫu & Khẩu vị:</b>"),
            horizon_slider,
            hc_strategy_radio,
            n_slider,
            force_refresh_checkbox,
            run_button,
            output_box
        ], layout=widgets.Layout(max_width='100%', overflow_x='hidden'))
    )

run_vietlott_council_app()


╔═══════════════════════════════════════════════════════════════════════╗
║         🏛️ VIETLOTT HARMONY COUNCIL ENGINE - MOBILE PRO v2.6          ║
║   [Developer: TrườngVN] | [Core: Đông - Tây - Thống kê - Smart Cache] ║
╚═══════════════════════════════════════════════════════════════════════╝

